In [ ]:
import pandas as pd
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_API_KEY",
)

df = pd.read_csv("./aax_results/aax_results-5-gepa.csv")
df.head()

In [ ]:
# Filter only rows where 'Result' is False
df_failures = df[df['Result'] == False]
df_failures.reset_index(drop=True, inplace=True)
df_failures

In [ ]:
# Normalize column names
df_failures.columns = [col.strip().lower().replace(' ', '_') for col in df_failures.columns]
json_failures = df_failures.to_dict(orient='records')
print(json_failures[0])

In [ ]:
def template():
    system_instructions = """
    You are given a dataset of failed customer-support agent turns.
    Each row is a normalized JSON object with fields such as:

    session_id, turn, question, expected_chunk_id, actual_chunk_id, found,
    expected_answer, actual_answer, result, comments, reason_for_failure

    Your task is to analyze ALL rows and produce:

    1) FAILURE PATTERNS
       - Group rows into recurring failure patterns.
       - For each pattern: name, frequency (%), common traits, root causes, example (session_id, turn).

    2) COMMONALITIES
       - Rank the most frequent: missing actions, retrieval errors, chunk mismatches,
         policy mistakes, hallucinations, formatting issues.

    3) OUTLIERS
       - Identify rare or unique failures and explain why they don’t fit main patterns,
         plus recommended handling.

    4) DECISION TREE
       - Produce a concise, hierarchical decision tree covering ~80% of failures.
       - Include intent classification, retrieval checks, chunk-validation logic,
         policy checks, fallback steps, and escalation rules.

    5) COVERAGE ANALYSIS
       - Total rows, rows covered by the tree, coverage %, uncovered gaps,
         and 3–5 concrete recommendations.

    Rules:
    - Use only information present in the rows (no invented policy facts).
    - Keep pattern names short.
    - Keep the decision tree actionable and directly implementable.
    - Produce in a human-readable form.
    """

    return {
        "role": "system",
        "content": system_instructions
    }

In [ ]:
system_instruction = template()
response = client.chat.completions.create(
    model="gpt-5",
    messages=[
        system_instruction,
        { "role": "user", "content": str(json_failures) }
    ]
)
print(response.choices[0].message.content)

In [ ]:
old_system_instruction = ""
with open("./system_instruction-3.txt", "r") as f:
    old_system_instruction = f.read()

combine_instruction = ""
with open("./test.md", "r") as f:
    combine_instruction = f.read()

decision_tree = """
4) DECISION TREE (actionable, ~80%+ coverage)

A. Classify intent (single best match; ask 1 clarifier if ambiguous)
- Benefits/limits lookup: dental, disability/rehab, child allowance, survivor, etc.
- Calculations/formulas: old-age pension/bene formula, payout type (monthly vs. lump sum).
- Procedures/documents: registration, reimbursement steps.
- Contact info: specific office numbers.
- Enrollment channels: where/how to sign up (e.g., M.40).
- Out-of-scope/general opinion: solvency, forecasts, commentary.

B. Retrieve evidence
- Build query with intent-specific keywords; fetch top N chunks (e.g., 3–5).
- Require at least one chunk containing clear, intent-relevant key phrases:
  - Benefits/limits: terms like limit/baht/percentage and the benefit name.
  - Calculations: “สูตร,” “คำนวณ,” “เปอร์เซ็นต์,” “รายเดือน/รายก้อน.”
  - Procedures/docs: form codes (e.g., สปส.), “เอกสาร,” “ขึ้นทะเบียน.”
  - Contact info: office name + phone pattern.
- If no such chunk is found, re-query with synonyms; if still none → go to Fallback E.

C. Chunk validation logic
- For numeric answers: verify all numbers are present in the retrieved chunk(s). Do not invent or infer.
- For “what to bring/what’s covered” questions: ensure the list includes all subitems explicitly mentioned in chunk(s).
- For continuity/conditions questions: verify condition phrases (e.g., “จนถึงอายุ …,” “รายเดือนตลอดชีวิต”) appear in chunk(s).

D. Assemble answer with intent-specific checklists
- Dental benefits: include basic care limit; include dentures (partial/full) with limits; include contracted-facility payment note if present.
- Old-age benefits:
  - If formula requested or context implies calculation: provide formula and explicitly state payout cadence/type (e.g., monthly for life vs. lump sum).
  - If 12–179 months context: provide lump-sum components (employee+employer+returns) and “paid once.”
- Child allowance continuity: explicitly state continuation condition and the age limit.
- Registration/procedures: match the exact role/topic (e.g., insured person ID vs. employer registration); list all required documents.
- Contact info: provide the specific office phone number if present; avoid substituting generic lines when a specific number is expected.
- Disability/rehab/private hospital: state only what's in the chunk; if rehab rights are mentioned but monthly caps are not, do not introduce caps.

E. Policy checks and safety rails
- No unsupported numbers or steps. If a detail is not in the retrieved chunk(s), omit it or ask a clarifier.
- If retrieved chunks disagree, ask a focused clarification or present both clearly labeled possibilities.
- If the question is out-of-scope or the right chunk cannot be found after re-query:
  - Respond with the standard fallback: “ขออภัยครับ ข้อมูลส่วนนี้ไม่ปรากฎในคู่มือประกันตนที่ผมเข้าถึง ณ ตอนนี้ จึงยืนยันคำตอบให้ไม่ได้ครับ”
  - Offer to narrow the question to topics covered by the available manual.

F. Escalation/clarification rules
- If intent ambiguity remains after one clarifier or required evidence is absent, use the fallback and propose a next step (e.g., ask for missing specifics like location/office name, or the exact benefit type).
- For contact numbers, if the office is unspecified, ask for the office name first; if still unknown, provide the fallback.

G. Final self-check before sending
- Is every numeric claim in the answer grounded verbatim in retrieved text?
- Did we cover all mandatory subitems for this intent’s checklist?
- Did we avoid switching to a different subtopic than asked?
- If any answer is partially grounded only, either explicitly state the limits of knowledge or apply the fallback.
"""

prompt = f"""
SYSTEM INSTRUCTION:
```
{old_system_instruction}
```
DECISION TREE:
```
{decision_tree}
```
"""


response = client.chat.completions.create(
    model="gpt-5",
    messages=[
        { "role": "system", "content": combine_instruction },
        { "role": "user", "content": prompt }
    ]
)
print(response.choices[0].message.content)